# 03 · Inference
Load the packaged pipeline and score patients — first locally, then through the running API (container or Kubernetes).

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))   # works even if `pip install -e .` was skipped

import json, urllib.request
from heartrisk import config
from heartrisk.predictor import HeartRiskPredictor
predictor = HeartRiskPredictor(config.MODEL_PATH)
print("Loaded", predictor.model_name, predictor.version)

## 1 · Direct inference with the saved pipeline

In [ ]:
patients = {
    "typical high-risk profile": dict(age=63, sex=1, cp=4, trestbps=150, chol=260, fbs=0, restecg=2,
                                      thalach=112, exang=1, oldpeak=3.2, slope=2, ca=2, thal=7),
    "typical low-risk profile":  dict(age=41, sex=0, cp=2, trestbps=120, chol=190, fbs=0, restecg=0,
                                      thalach=178, exang=0, oldpeak=0.0, slope=1, ca=0, thal=3),
    "optional fields missing":   dict(age=55, sex=1, cp=3, trestbps=135, chol=230, fbs=0, restecg=0,
                                      thalach=150, exang=0, oldpeak=1.0, slope=2, ca=None, thal=None),
}
for name, rec in patients.items():
    print(f"{name:28s}", predictor.predict([rec])[0])

## 2 · Through the REST API
Start it first with `make docker-run` (or `make serve`, or deploy to Kubernetes).

In [ ]:
BASE = "http://localhost:8000"      # Kubernetes via Docker Desktop LoadBalancer: http://localhost
def call(rec):
    req = urllib.request.Request(f"{BASE}/predict", data=json.dumps(rec).encode(),
                                 headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=10) as r:
        return json.load(r)
try:
    print(json.dumps(call(patients["typical high-risk profile"]), indent=2))
except Exception as exc:
    print("API not reachable:", exc)